We use this notebook to perform an initial data merge.

In [78]:
import pandas as pd
financial_data = pd.read_csv("csv_data/compustat_large_cap_v2.csv")
ceo_data = pd.read_csv("csv_data/CEOS.csv")
ceo_controls = pd.read_csv("csv_data/CEO_CONTROLS.csv")
bridge = pd.read_csv("csv_data/boardex_id_data.csv")
education_organized = pd.read_csv("csv_data/EDUCATION_ORGANIZED.csv")

In [79]:
bridge.head(1)

,ticker,boardname,boardid
0,ATVI,ACTIVISION BLIZZARD INC (De-listed 10/2023),725


In [80]:
financial_data = financial_data.merge(
    bridge[['ticker', 'boardid']], 
    left_on='tic', 
    right_on='ticker', 
    how='left'
)
financial_data.drop(columns=['ticker'], inplace=True)
financial_data.head(1)

,costat,curcd,datafmt,indfmt,consol,sic,datadate,gvkey,conm,tic,...,lse,ni,revt,xrd,csho,prcc_f,sich,mkt_cap,industry,boardid
0,A,USD,STD,INDL,C,3674,12/31/15,1161,ADVANCED MICRO DEVICES,AMD,...,3109.0,-660.0,3991.0,947.0,792.0,2.87,3674.0,2273.04,Semiconductors/Hardware,881.0


In [81]:
financial_data = financial_data.dropna(subset=['boardid'])

In [82]:
print(ceo_data.columns)

Index(['companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority'],
      dtype='object')


In [83]:
# Ensure date columns are datetime
ceo_data['dateendrole'] = ceo_data['dateendrole'].replace('9000-01-01', '2027-01-01')

ceo_data['datestartrole'] = pd.to_datetime(ceo_data['datestartrole'], errors='coerce')
ceo_data['dateendrole'] = pd.to_datetime(ceo_data['dateendrole'], errors='coerce')

ceo_data['startyear'] = ceo_data['datestartrole'].dt.year
ceo_data['endyear'] = ceo_data['dateendrole'].dt.year

# Merge on boardid == companyid, then filter where fyear is strictly between start and end year
merged = financial_data.merge(
    ceo_data,
    left_on='boardid',
    right_on='companyid',
    how='left'
)

merged = merged[
    (merged['fyear'] >= merged['startyear']) &
    (merged['fyear'] < merged['endyear'])
]

# Drop helper columns if you don't need them
merged = merged.drop(columns=['startyear', 'endyear'])

print(merged.shape)
print(merged['gvkey'].nunique())
print(merged['fyear'].value_counts().sort_index())

(1097, 34)
161
fyear
2014     15
2015     88
2016     85
2017     90
2018     90
2019     90
2020    102
2021    114
2022    109
2023    112
2024    104
2025     98
Name: count, dtype: int64


In [84]:
merged = merged[merged['fyear'] >= 2015]
print(merged.shape)

(1082, 34)


In [85]:
print(merged['industry'].value_counts())
print(merged['gvkey'].nunique())

industry
Semiconductors/Hardware    418
Biotech                    400
Software                   264
Name: count, dtype: int64
160


In [86]:
merged = merged.merge(
    education_organized,
    on='directorid',
    how='left'
)

In [87]:
merged.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'MBA', 'Master's', 'PhD', 'MD'],
      dtype='object')

In [88]:
edu_cols = ['UG', 'MBA', "Master's", 'PhD', 'MD']
print(merged[edu_cols].isna().sum())
print(f"\nRows where ALL education fields are NaN: {merged[edu_cols].isna().all(axis=1).sum()}")

UG           205
MBA          687
Master's     743
PhD          897
MD          1036
dtype: int64

Rows where ALL education fields are NaN: 85


In [89]:
len(merged)

1082